# 01. Data Preprocessing and Temporal Split

**Objective:** Load raw CVE data from JSON files, clean the dataset, and perform a strict time-based train/validation/test split. This ensures temporal isolation and prevents future information from leaking into the training phase, which is critical for the Temporal-Contextual Threat Re-Ranking Framework (TCTR).

**Inputs:** `processed_cves_2020.json` through `processed_cves_2026.json`
**Outputs:** Chronologically split Train, Validation, and Test datasets saved as Parquet files.

In [17]:
import pandas as pd
import json
import os
import glob
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

# Set up paths relative to the notebooks directory
# Working directory is D:\NetShield\NetShieldAI\notebooks
DATA_DIR = os.path.join("..", "Data", "Processed_CVEs")
OUTPUT_DIR = os.path.join("..", "Data", "TCTR_splits")

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Data directory configured: {os.path.abspath(DATA_DIR)}")
print(f"Output directory configured: {os.path.abspath(OUTPUT_DIR)}")

Data directory configured: d:\NetShield\NetShieldAI\Data\Processed_CVEs
Output directory configured: d:\NetShield\NetShieldAI\Data\TCTR_splits


## 1. Load Raw CVE Data

Iteratively load all JSON files from the `Processed_CVEs` directory.

In [18]:
# Locate all processed JSON files from 2020 to 2026
file_pattern = os.path.join(DATA_DIR, "processed_cves_*.json")
json_files = sorted(glob.glob(file_pattern))

if not json_files:
    print("Warning: No JSON files found. Please check your DATA_DIR.")

data_frames = []

for file in json_files:
    print(f"Loading {os.path.basename(file)}...")
    with open(file, 'r', encoding='utf-8') as f:
        cve_list = json.load(f)
        # Convert JSON array of dicts into a DataFrame
        df_chunk = pd.DataFrame(cve_list)
        data_frames.append(df_chunk)

# Concatenate all years into a single DataFrame
df_raw = pd.concat(data_frames, ignore_index=True)
print(f"\nTotal CVE records loaded: {len(df_raw)}")

Loading processed_cves_2020.json...
Loading processed_cves_2021.json...
Loading processed_cves_2022.json...
Loading processed_cves_2023.json...
Loading processed_cves_2024.json...
Loading processed_cves_2025.json...
Loading processed_cves_2026.json...

Total CVE records loaded: 183833


## 2. Data Cleaning and Formatting

Ensure date columns are in datetime format and handle missing values.

In [19]:
df = df_raw.copy()

# 1. Drop records missing critical identifiers or dates
initial_len = len(df)
df = df.dropna(subset=['cve_id', 'published_date'])
print(f"Dropped {initial_len - len(df)} rows missing CVE IDs or Published Dates.")

# 2. Convert published_date and last_modified_date to datetime objects (UTC)
df['published_date'] = pd.to_datetime(df['published_date'], errors='coerce', utc=True)
df['last_modified_date'] = pd.to_datetime(df['last_modified_date'], errors='coerce', utc=True)

# Drop rows where published_date failed to parse
df = df.dropna(subset=['published_date'])

# 3. Handle list columns (keywords, platforms, affected_products)
# Ensuring they are converted to standard Python lists and replacing nulls with empty lists
list_cols = ['keywords', 'platforms', 'affected_products']
for col in list_cols:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: x if isinstance(x, list) else [])

# 4. Sort strictly by publication date to prepare for temporal splitting
df = df.sort_values(by='published_date').reset_index(drop=True)

print("\nData schema after cleaning:")
print(df.dtypes)
print(f"\nTotal cleaned records: {len(df)}")

Dropped 0 rows missing CVE IDs or Published Dates.

Data schema after cleaning:
cve_id                                 object
description                            object
keywords                               object
platforms                              object
affected_products                      object
published_date            datetime64[ns, UTC]
last_modified_date        datetime64[ns, UTC]
cwe_id                                 object
base_score                            float64
severity                               object
attack_vector                          object
attack_complexity                      object
privileges_required                    object
user_interaction                       object
scope                                  object
confidentiality_impact                 object
integrity_impact                       object
availability_impact                    object
exploitability_score                   object
impact_score                           object


## 3. Temporal Splitting

Split the data chronologically:
- **Train:** 2020 - 2024
- **Validation:** 2025
- **Test:** 2026

In [21]:
# Adjusted temporal boundaries to ensure a populated Test Set
# Train: 2020 - 2023
# Validation: 2024
# Test: 2025 onwards

train_end_date = pd.to_datetime("2023-12-31 23:59:59", utc=True)
val_end_date   = pd.to_datetime("2024-12-31 23:59:59", utc=True)

# Split the data
train_df = df[df['published_date'] <= train_end_date].copy()
val_df   = df[(df['published_date'] > train_end_date) & (df['published_date'] <= val_end_date)].copy()
test_df  = df[df['published_date'] > val_end_date].copy()

print("Adjusted Temporal Split Complete:")
print(f"Training Set (<= 2023):  {len(train_df)} records")
print(f"Validation Set (2024):   {len(val_df)} records")
print(f"Test Set (>= 2025):      {len(test_df)} records")

Adjusted Temporal Split Complete:
Training Set (<= 2023):  90566 records
Validation Set (2024):   40484 records
Test Set (>= 2025):      6976 records


## 4. Save Processed Datasets

Save the split datasets as Parquet files for efficient loading in subsequent notebooks.

In [22]:
# Verify that strict chronological order is maintained across splits
assert train_df['published_date'].max() < val_df['published_date'].min(), "Leakage detected between Train and Validation!"
assert val_df['published_date'].max() < test_df['published_date'].min(), "Leakage detected between Validation and Test!"

print("Sanity Check Passed: No temporal leakage detected across dataset boundaries.")

# Display date ranges
print("\nDataset Date Ranges:")
print(f"Train: {train_df['published_date'].min().date()} to {train_df['published_date'].max().date()}")
print(f"Val:   {val_df['published_date'].min().date()} to {val_df['published_date'].max().date()}")
print(f"Test:  {test_df['published_date'].min().date()} to {test_df['published_date'].max().date()}")

Sanity Check Passed: No temporal leakage detected across dataset boundaries.

Dataset Date Ranges:
Train: 2020-01-02 to 2023-12-31
Val:   2024-01-01 to 2024-12-31
Test:  2025-01-01 to 2025-06-02


In [23]:
# Save datasets to Parquet format (preserves column types and list structures better than CSV)
train_path = os.path.join(OUTPUT_DIR, "train_cve.parquet")
val_path   = os.path.join(OUTPUT_DIR, "val_cve.parquet")
test_path  = os.path.join(OUTPUT_DIR, "test_cve.parquet")

train_df.to_parquet(train_path, index=False)
val_df.to_parquet(val_path, index=False)
test_df.to_parquet(test_path, index=False)

print("Datasets successfully saved to Parquet files:")
print(f"- {os.path.abspath(train_path)}")
print(f"- {os.path.abspath(val_path)}")
print(f"- {os.path.abspath(test_path)}")

Datasets successfully saved to Parquet files:
- d:\NetShield\NetShieldAI\Data\TCTR_splits\train_cve.parquet
- d:\NetShield\NetShieldAI\Data\TCTR_splits\val_cve.parquet
- d:\NetShield\NetShieldAI\Data\TCTR_splits\test_cve.parquet
